# LingkodAI demo: text chat

Type a question about PRC services in Cebuano, Filipino or English and get
the answer in the same language, with every stage shown: the language
detected, the question in English, the knowledge-base entries used, the
English answer, the translated answer and, optionally, the answer spoken.

Typed text stands in for speech, so this skips stages 1 and 2 (audio
language ID and speech recognition). The language comes from GlotLID on the
first question and stays fixed for the conversation, as in the full
pipeline. For spoken questions, use `demo_audio.ipynb`.

**Before you start**

- Runtime: GPU, **L4** (24 GB). Without speech (`SPEAK = False`) a smaller
  GPU may do; that is untested.
- For speech: a Colab Secret named `HF_TOKEN` (key icon in the left sidebar,
  notebook access on) that can read `lingkodai/lingkodai-eval-assets`.
  Text-only chat needs no token.
- The first question downloads and loads the models (roughly 30 GB on a
  fresh runtime) and can take 10 minutes or more. Later questions take
  seconds to tens of seconds.

**Voice notice.** Synthesized audio here uses a cloned voice from a Philippine
Languages Database research speaker, under a CC-BY-NC, research-only licence.
It is not a real PRC official. Keep the audio and any saved sessions private.

## 1. Setup

On Colab this clones the `demo-notebooks` branch and installs the pipeline (a few
minutes), then reads the `HF_TOKEN` Secret. On your own machine it uses your
LingkodAI checkout (`~/LingkodAI`, or set `LINGKODAI_DIR`) with its
environment already installed, and reads its `.env`.

In [ ]:
BRANCH = "demo-notebooks"
import os, re, subprocess, sys
def run(cmd, log=None):
    """Run a command and print its output here; raise if it fails. With log=,
    everything also goes to that file and compile progress lines are skipped."""
    cmd = [str(c) for c in cmd]
    print("$", " ".join(cmd), flush=True)
    sink = open(log, "w", encoding="utf-8") if log else None
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in proc.stdout:
        if sink:
            sink.write(line)
        if not (sink and re.match(r"^\[ *\d+%\]", line)):
            print(line, end="", flush=True)
    if sink:
        sink.close()
    if proc.wait() != 0:
        raise RuntimeError(f"exit code {proc.returncode}: {' '.join(cmd)}" + (f" (full log: {log})" if log else ""))
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    REPO = "/content/LingkodAI"
    if not os.path.isdir(REPO):
        run(["git", "clone", "-q", "--branch", BRANCH, "https://github.com/isabelbeab/LingkodAI.git", REPO])
    os.chdir(REPO)
    run([sys.executable, "-m", "pip", "install", "-q", "-e", ".[gpu,demo]"])
    from google.colab import userdata
    try:
        os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    except Exception as exc:  # reported, not hidden: later steps that need it check again
        print(f"HF_TOKEN Secret not read ({type(exc).__name__}): fine for text-only chat; speech needs it")
else:
    REPO = os.path.abspath(os.path.expanduser(os.environ.get("LINGKODAI_DIR", "~/LingkodAI")))
    os.chdir(REPO)
if REPO not in sys.path:
    sys.path.insert(0, REPO)
from src import demo_ui
if not IN_COLAB:
    print(".env set:", demo_ui.load_env_file(".env") or "nothing new")

import torch, transformers
assert transformers.__version__ == "4.57.3", (
    f"transformers {transformers.__version__} is loaded, not 4.57.3: restart the runtime (or kernel) and run from step 1")
run(["git", "log", "--oneline", "-1"])
print("Colab" if IN_COLAB else "local", "|", REPO, "| torch", torch.__version__, "| transformers", transformers.__version__)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")

## 2. Settings

Change these before running the next steps.

In [ ]:
SPEAK = True          # speak each answer (needs HF_TOKEN for the reference voices)
TTS_BACKEND = "cpp"   # "cpp": qwentts.cpp, fast | "hf": qwen-tts, the evaluated path, slow
QUANT = "Q8_0"        # cpp weights: "Q8_0", or "F32" (about 5 GB more GPU memory)
KEEP_LOADED = True    # False on a smaller GPU: frees each model after use (much slower turns)

## 3. Voice

Skipped when `SPEAK` is False. Fetches the three reference voices from the
private eval assets. For `TTS_BACKEND = "cpp"` it also builds the patched
`tts-server`, or restores an earlier build from the cache (on your Drive on Colab, `outputs/qwentts_cache/` locally; the first
build takes several minutes), and downloads the `QUANT` weights with their
checksums checked.

In [ ]:
ASSETS = f"{REPO}/outputs/eval_assets"
TTS_OPTIONS = None
if not SPEAK:
    print("SPEAK is False: no voice setup")
else:
    assert os.environ.get("HF_TOKEN"), "Speech needs HF_TOKEN: the reference voices are in the private eval assets"
    run([sys.executable, "scripts/fetch_eval_assets.py", "--out", ASSETS])
    os.environ["TTS_REF_DIR"] = f"{ASSETS}/tts_ref"
    if TTS_BACKEND == "cpp":
        if IN_COLAB:
            from google.colab import drive
            drive.mount("/content/drive")
            cache, gguf = "/content/drive/MyDrive/lingkodai_cache", "/content/models/gguf"
        else:
            cache, gguf = f"{REPO}/outputs/qwentts_cache", f"{REPO}/outputs/gguf"
        run(["bash", "scripts/build_qwentts.sh", "--cuda-arch", "native", "--cache-dir", cache],
            log=f"{REPO}/outputs/build_qwentts.log")
        run([sys.executable, "scripts/fetch_qwentts_weights.py", "--out", gguf, "--quant", QUANT])
        TTS_OPTIONS = {"mode": "segmented", "log_dir": f"{REPO}/outputs/demo_tts",
                       "talker_gguf": f"{gguf}/qwen-talker-1.7b-base-{QUANT}.gguf",
                       "codec_gguf": f"{gguf}/qwen-tokenizer-12hz-{QUANT}.gguf"}
    print("Voice ready:", TTS_BACKEND, QUANT if TTS_BACKEND == "cpp" else "")

## 4. Start the session

Nothing loads yet: each model loads the first time a question needs it and
then stays loaded, so the first question is slow (it also downloads the
models on a fresh runtime) and later ones are not. Free GPU memory prints after
every load.

In [ ]:
from src.demo import DemoSession, gpu_memory

session = DemoSession(tts_backend=TTS_BACKEND, tts_options=TTS_OPTIONS, keep_loaded=KEEP_LOADED)


def ask(question, lang=None, speak=None):
    """Ask in the current conversation and show every stage. lang= skips
    language detection on a conversation's first question."""
    session.ask_text(question, lang=lang, speak=SPEAK if speak is None else speak).show()


print("Ready. GPU", gpu_memory())

## 5. Ask

Each `ask(...)` is one turn. A follow-up is read in the light of the
conversation so far (the "Follow-up read as" row shows the rewritten
question). `session.reset()` starts a new conversation, and the next
question sets the language again.

In [ ]:
ask("Unsa ang mga kinahanglanon aron ma-renew ang akong PRC ID?")

In [ ]:
ask("Pila ang bayad?")

In [ ]:
session.reset()
ask("Ano ang mga kailangan para makakuha ng certified true copy ng aking rating?")

In [ ]:
session.reset()
ask("How do I renew my PRC ID?")

## 6. Chat box

The same session as a chat window. Tick "Show every stage" for the full
breakdown under each answer. "New conversation" is `session.reset()`.

In [ ]:
demo_ui.ChatBox(session, speak=SPEAK)

## 7. Hear any text

Any sentence in the cloned voice, outside the conversation. The language must
be `"ceb"`, `"fil"` or `"eng"`.

In [ ]:
from IPython.display import Audio

result = session.speak("Maayong adlaw! Ako si LingkodAI.", "ceb")
Audio(result.audio, rate=result.sr) if result.ok else print("text only:", result.failure_reason)

## 8. Save this conversation

Writes every turn's text and audio to
`MyDrive/lingkodai_demo_sessions/` on Colab, or `outputs/demo_sessions/`
locally, for `demo_replay.ipynb`. Keep the folder private.

In [ ]:
import time
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    SAVE_ROOT = "/content/drive/MyDrive/lingkodai_demo_sessions"
else:
    SAVE_ROOT = f"{REPO}/outputs/demo_sessions"
session.save(f"{SAVE_ROOT}/chat-{time.strftime('%Y%m%d-%H%M%S')}")

## 9. Free the GPU

Unloads every model and stops the TTS server. Run step 4 again to start over.

In [ ]:
session.unload()